# The Pandas 3 Trap | Schema-Aware AutoML Agent

**A clean-room agent archive, a 16-task offline audit, and the exact bug that can turn categorical learning into random guessing.**

`clean-room code` · `deterministic portfolio` · `pandas 3 safe` · `submission.zip included`

> Evidence boundary: every score below is an **offline replay on the 16 official local tasks**. It is not an official Kaggle leaderboard score. Public notebook titles and Kaggle sort order are not treated as verified scores.

In [ ]:
from pathlib import Path
from IPython.display import HTML, Markdown, display
import hashlib, json, os, py_compile, shutil, subprocess, sys, tempfile, textwrap, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
COLORS = {"ink":"#17243D", "blue":"#176BFF", "cyan":"#26C6DA", "gold":"#FFB000", "red":"#E84A5F", "green":"#00A676", "mist":"#EEF4FF"}
plt.rcParams.update({"figure.dpi": 130, "axes.titleweight": "bold", "axes.edgecolor": "#CAD5E5", "font.family": "DejaVu Sans"})

In [ ]:
BASELINE = json.loads("[{\"best_public_auc\": 0.7218037678, \"dataset\": \"train_01\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.7131237298, \"oracle_best_private_auc\": 0.7131237298, \"selected\": \"['logistic', 'rank_top2']\", \"selected_private_auc\": \"[0.7127709590180986, 0.7131237298170937]\", \"selected_public_auc\": \"[0.7218037678176455, 0.721058485909724]\"}, {\"best_public_auc\": 0.9702986757, \"dataset\": \"train_02\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.9688542467, \"oracle_best_private_auc\": 0.9688542467, \"selected\": \"['rank_top2', 'lightgbm']\", \"selected_private_auc\": \"[0.968854246671787, 0.9657341269841271]\", \"selected_public_auc\": \"[0.9702986756814878, 0.9681170254162655]\"}, {\"best_public_auc\": 0.8086873273, \"dataset\": \"train_03\", \"final_candidate\": \"rank_all\", \"final_private_auc\": 0.8201704696, \"oracle_best_private_auc\": 0.8209036689, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.8193281385314635, 0.8201704696379961]\", \"selected_public_auc\": \"[0.8086873272953823, 0.8077170051673174]\"}, {\"best_public_auc\": 0.8346731578, \"dataset\": \"train_04\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8358894289, \"oracle_best_private_auc\": 0.8358894289, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.8358894289428943, 0.8341667766776677]\", \"selected_public_auc\": \"[0.8346731578260067, 0.8346674766479578]\"}, {\"best_public_auc\": 0.6803083413, \"dataset\": \"train_05\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.6898729235, \"oracle_best_private_auc\": 0.6898729235, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.6898729234581834, 0.6887150417088674]\", \"selected_public_auc\": \"[0.6803083413399418, 0.6798526976302521]\"}, {\"best_public_auc\": 0.8016230883, \"dataset\": \"train_06\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8149850743, \"oracle_best_private_auc\": 0.8149850743, \"selected\": \"['catboost', 'rank_top2']\", \"selected_private_auc\": \"[0.8145053692560819, 0.8149850743306813]\", \"selected_public_auc\": \"[0.8016230882596942, 0.8015584482493516]\"}, {\"best_public_auc\": 0.8323653836, \"dataset\": \"train_07\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8328690716, \"oracle_best_private_auc\": 0.8328690716, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.8328690716414524, 0.8298441690667924]\", \"selected_public_auc\": \"[0.8323653836151885, 0.8299924385179496]\"}, {\"best_public_auc\": 0.853825521, \"dataset\": \"train_08\", \"final_candidate\": \"rank_all\", \"final_private_auc\": 0.8488367737, \"oracle_best_private_auc\": 0.8490670894, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.848633836473456, 0.8488367736589936]\", \"selected_public_auc\": \"[0.8538255209933829, 0.8538143975477844]\"}, {\"best_public_auc\": 0.6391989232, \"dataset\": \"train_09\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.6578476177, \"oracle_best_private_auc\": 0.6578476177, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.6578476177279069, 0.656349031987697]\", \"selected_public_auc\": \"[0.6391989232264353, 0.638017781614101]\"}, {\"best_public_auc\": 0.8448426952, \"dataset\": \"train_10\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.857216387, \"oracle_best_private_auc\": 0.857216387, \"selected\": \"['catboost', 'rank_top2']\", \"selected_private_auc\": \"[0.8561713664190653, 0.8572163870370397]\", \"selected_public_auc\": \"[0.8448426951748313, 0.8446102151376345]\"}, {\"best_public_auc\": 0.8334757805, \"dataset\": \"train_11\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8196257026, \"oracle_best_private_auc\": 0.8196257026, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.8196257025921222, 0.8174658141743498]\", \"selected_public_auc\": \"[0.8334757805123938, 0.8313256308908482]\"}, {\"best_public_auc\": 0.7961631465, \"dataset\": \"train_12\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.7731135792, \"oracle_best_private_auc\": 0.7731135792, \"selected\": \"['lightgbm', 'rank_top2']\", \"selected_private_auc\": \"[0.7726160578971082, 0.7731135791707627]\", \"selected_public_auc\": \"[0.796163146474931, 0.7958777060638969]\"}, {\"best_public_auc\": 0.6346082984, \"dataset\": \"train_13\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.6433161733, \"oracle_best_private_auc\": 0.6433161733, \"selected\": \"['catboost', 'rank_all']\", \"selected_private_auc\": \"[0.643316173264693, 0.6431532926131704]\", \"selected_public_auc\": \"[0.6346082984331938, 0.6338822155288621]\"}, {\"best_public_auc\": 0.8017251724, \"dataset\": \"train_14\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.811982704, \"oracle_best_private_auc\": 0.811982704, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.8111515741542605, 0.8119827040492698]\", \"selected_public_auc\": \"[0.8017251724164414, 0.8007254098570492]\"}, {\"best_public_auc\": 0.8630496291, \"dataset\": \"train_15\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8678123336, \"oracle_best_private_auc\": 0.8678123336, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.8678123336007725, 0.8640316151130152]\", \"selected_public_auc\": \"[0.8630496291455325, 0.8590210415233115]\"}, {\"best_public_auc\": 0.9113972197, \"dataset\": \"train_16\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.9071317803, \"oracle_best_private_auc\": 0.9071317803, \"selected\": \"['catboost', 'rank_top2']\", \"selected_private_auc\": \"[0.9071317803162186, 0.9024574326192476]\", \"selected_public_auc\": \"[0.9113972197219722, 0.9054264626462647]\"}]")
RAW_BASELINE = json.loads("[{\"best_public_auc\": 0.7091450154, \"dataset\": \"train_01\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.7050047207, \"oracle_best_private_auc\": 0.7050047207, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.7050047206797779, 0.7041489174441119]\", \"selected_public_auc\": \"[0.7091450154112394, 0.7084422936122716]\"}, {\"best_public_auc\": 0.9699182669, \"dataset\": \"train_02\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.968359975, \"oracle_best_private_auc\": 0.968359975, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.9683599750384024, 0.9668225166410651]\", \"selected_public_auc\": \"[0.9699182669168698, 0.9677816576893932]\"}, {\"best_public_auc\": 0.7776647015, \"dataset\": \"train_03\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.7905067812, \"oracle_best_private_auc\": 0.7905067812, \"selected\": \"['catboost', 'rank_all']\", \"selected_private_auc\": \"[0.790506781213456, 0.7891663162287632]\", \"selected_public_auc\": \"[0.7776647015403384, 0.7758845508683855]\"}, {\"best_public_auc\": 0.8346731578, \"dataset\": \"train_04\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8358894289, \"oracle_best_private_auc\": 0.8358894289, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.8358894289428943, 0.8341667766776677]\", \"selected_public_auc\": \"[0.8346731578260067, 0.8346674766479578]\"}, {\"best_public_auc\": 0.5738107563, \"dataset\": \"train_05\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.5568932731, \"oracle_best_private_auc\": 0.5568932731, \"selected\": \"['rank_all', 'rank_top2']\", \"selected_private_auc\": \"[0.5565200080462019, 0.5568932730839021]\", \"selected_public_auc\": \"[0.5738107562996668, 0.5737481934063358]\"}, {\"best_public_auc\": 0.5, \"dataset\": \"train_06\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.5, \"oracle_best_private_auc\": 0.5, \"selected\": \"['catboost', 'lightgbm']\", \"selected_private_auc\": \"[0.5, 0.5]\", \"selected_public_auc\": \"[0.5, 0.5]\"}, {\"best_public_auc\": 0.7528384363, \"dataset\": \"train_07\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.7562754946, \"oracle_best_private_auc\": 0.7562754946, \"selected\": \"['rank_all', 'rank_top2']\", \"selected_private_auc\": \"[0.7555301313199643, 0.7562754945843337]\", \"selected_public_auc\": \"[0.7528384363335214, 0.7525463790903016]\"}, {\"best_public_auc\": 0.5, \"dataset\": \"train_08\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.5, \"oracle_best_private_auc\": 0.5, \"selected\": \"['catboost', 'lightgbm']\", \"selected_private_auc\": \"[0.5, 0.5]\", \"selected_public_auc\": \"[0.5, 0.5]\"}, {\"best_public_auc\": 0.6332772127, \"dataset\": \"train_09\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.6528638261, \"oracle_best_private_auc\": 0.6528638261, \"selected\": \"['rank_all', 'catboost']\", \"selected_private_auc\": \"[0.6527382172086063, 0.6528638260715676]\", \"selected_public_auc\": \"[0.6332772127042254, 0.6328149016192729]\"}, {\"best_public_auc\": 0.8448426952, \"dataset\": \"train_10\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.857216387, \"oracle_best_private_auc\": 0.857216387, \"selected\": \"['catboost', 'rank_top2']\", \"selected_private_auc\": \"[0.8561713664190653, 0.8572163870370397]\", \"selected_public_auc\": \"[0.8448426951748313, 0.8446102151376345]\"}, {\"best_public_auc\": 0.8334757805, \"dataset\": \"train_11\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8196257026, \"oracle_best_private_auc\": 0.8196257026, \"selected\": \"['rank_top2', 'rank_all']\", \"selected_private_auc\": \"[0.8196257025921222, 0.8174658141743498]\", \"selected_public_auc\": \"[0.8334757805123938, 0.8313256308908482]\"}, {\"best_public_auc\": 0.7739876745, \"dataset\": \"train_12\", \"final_candidate\": \"lightgbm\", \"final_private_auc\": 0.7532521683, \"oracle_best_private_auc\": 0.7539715302, \"selected\": \"['lightgbm', 'rank_all']\", \"selected_private_auc\": \"[0.7532521683255509, 0.7516777642950766]\", \"selected_public_auc\": \"[0.7739876745422514, 0.7739332744639151]\"}, {\"best_public_auc\": 0.5063154653, \"dataset\": \"train_13\", \"final_candidate\": \"lightgbm\", \"final_private_auc\": 0.4988344753, \"oracle_best_private_auc\": 0.5099113996, \"selected\": \"['extra_trees', 'lightgbm']\", \"selected_private_auc\": \"[0.4973223892895572, 0.4988344753379013]\", \"selected_public_auc\": \"[0.506315465261861, 0.5009960839843359]\"}, {\"best_public_auc\": 0.7479706348, \"dataset\": \"train_14\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.7445568513, \"oracle_best_private_auc\": 0.7445568513, \"selected\": \"['rank_top2', 'catboost']\", \"selected_private_auc\": \"[0.7443930735548357, 0.7445568512690333]\", \"selected_public_auc\": \"[0.7479706348048251, 0.7472707130130253]\"}, {\"best_public_auc\": 0.6194839009, \"dataset\": \"train_15\", \"final_candidate\": \"rank_all\", \"final_private_auc\": 0.6322377612, \"oracle_best_private_auc\": 0.6322377612, \"selected\": \"['catboost', 'rank_all']\", \"selected_private_auc\": \"[0.6280238745830566, 0.6322377612192791]\", \"selected_public_auc\": \"[0.6194839009353834, 0.6164206441546585]\"}, {\"best_public_auc\": 0.9113972197, \"dataset\": \"train_16\", \"final_candidate\": \"catboost\", \"final_private_auc\": 0.9071317803, \"oracle_best_private_auc\": 0.9071317803, \"selected\": \"['catboost', 'rank_top2']\", \"selected_private_auc\": \"[0.9071317803162186, 0.9024574326192476]\", \"selected_public_auc\": \"[0.9113972197219722, 0.9054264626462647]\"}]")
CLEANROOM = json.loads("[{\"best_public_auc\": 0.7216405674, \"dataset\": \"train_01\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.7126573427, \"oracle_best_private_auc\": 0.7126573427, \"oracle_candidate\": \"rank_top2\", \"selected\": \"logistic+rank_top2\", \"selected_private_auc\": \"0.712408346802+0.712657342657\", \"selected_public_auc\": \"0.721640567400+0.720361124124\"}, {\"best_public_auc\": 0.9705476414, \"dataset\": \"train_02\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.9689826549, \"oracle_best_private_auc\": 0.9689826549, \"oracle_candidate\": \"rank_top2\", \"selected\": \"catboost_ordinal+rank_top2\", \"selected_private_auc\": \"0.968844406042+0.968982654890\", \"selected_public_auc\": \"0.970547641418+0.970416438395\"}, {\"best_public_auc\": 0.8102336182, \"dataset\": \"train_03\", \"final_candidate\": \"rank_top3\", \"final_private_auc\": 0.8230879052, \"oracle_best_private_auc\": 0.8230879052, \"oracle_candidate\": \"rank_top3\", \"selected\": \"rank_top3+rank_top2\", \"selected_private_auc\": \"0.823087905188+0.821337555870\", \"selected_public_auc\": \"0.810233618173+0.807107753600\"}, {\"best_public_auc\": 0.8347938228, \"dataset\": \"train_04\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8367116712, \"oracle_best_private_auc\": 0.8367116712, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+catboost_native\", \"selected_private_auc\": \"0.836711671167+0.833120592059\", \"selected_public_auc\": \"0.834793822847+0.833464987300\"}, {\"best_public_auc\": 0.6831279973, \"dataset\": \"train_05\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.6868203916, \"oracle_best_private_auc\": 0.6868203916, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+rank_top3\", \"selected_private_auc\": \"0.686820391585+0.685310368479\", \"selected_public_auc\": \"0.683127997271+0.680587591335\"}, {\"best_public_auc\": 0.8020350083, \"dataset\": \"train_06\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8154932656, \"oracle_best_private_auc\": 0.8154932656, \"oracle_candidate\": \"rank_top2\", \"selected\": \"hist_gradient_boosting+rank_top2\", \"selected_private_auc\": \"0.814806475210+0.815493265645\", \"selected_public_auc\": \"0.802035008326+0.801957008313\"}, {\"best_public_auc\": 0.8332486767, \"dataset\": \"train_07\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8328678714, \"oracle_best_private_auc\": 0.8328678714, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+rank_top3\", \"selected_private_auc\": \"0.832867871379+0.832839065069\", \"selected_public_auc\": \"0.833248676741+0.832135338526\"}, {\"best_public_auc\": 0.8548148675, \"dataset\": \"train_08\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8502486081, \"oracle_best_private_auc\": 0.8502486081, \"oracle_candidate\": \"rank_top2\", \"selected\": \"logistic+rank_top2\", \"selected_private_auc\": \"0.849808790743+0.850248608059\", \"selected_public_auc\": \"0.854814867453+0.854665941322\"}, {\"best_public_auc\": 0.640452476, \"dataset\": \"train_09\", \"final_candidate\": \"rank_top3\", \"final_private_auc\": 0.6554791306, \"oracle_best_private_auc\": 0.6554791306, \"oracle_candidate\": \"rank_top3\", \"selected\": \"rank_top3+rank_top2\", \"selected_private_auc\": \"0.655479130607+0.653347460197\", \"selected_public_auc\": \"0.640452475973+0.639898390777\"}, {\"best_public_auc\": 0.8466805355, \"dataset\": \"train_10\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8574972248, \"oracle_best_private_auc\": 0.8574972248, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+catboost_native\", \"selected_private_auc\": \"0.857497224827+0.856403397641\", \"selected_public_auc\": \"0.846680535469+0.845716295315\"}, {\"best_public_auc\": 0.8357659503, \"dataset\": \"train_11\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8231533122, \"oracle_best_private_auc\": 0.8231533122, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+rank_top3\", \"selected_private_auc\": \"0.823153312213+0.821169532109\", \"selected_public_auc\": \"0.835765950297+0.833109167720\"}, {\"best_public_auc\": 0.7953025852, \"dataset\": \"train_12\", \"final_candidate\": \"rank_top3\", \"final_private_auc\": 0.7723861373, \"oracle_best_private_auc\": 0.7728086984, \"oracle_candidate\": \"rank_top2\", \"selected\": \"hist_gradient_boosting+rank_top3\", \"selected_private_auc\": \"0.770618692784+0.772386137309\", \"selected_public_auc\": \"0.795302585236+0.795218985115\"}, {\"best_public_auc\": 0.6397187189, \"dataset\": \"train_13\", \"final_candidate\": \"rank_top3\", \"final_private_auc\": 0.6508653235, \"oracle_best_private_auc\": 0.6511157245, \"oracle_candidate\": \"hist_gradient_boosting\", \"selected\": \"catboost_native+rank_top3\", \"selected_private_auc\": \"0.648871235485+0.650865323461\", \"selected_public_auc\": \"0.639718718875+0.637730310921\"}, {\"best_public_auc\": 0.8026025347, \"dataset\": \"train_14\", \"final_candidate\": \"rank_top3\", \"final_private_auc\": 0.8110274807, \"oracle_best_private_auc\": 0.8114780095, \"oracle_candidate\": \"catboost_native\", \"selected\": \"hist_gradient_boosting+rank_top3\", \"selected_private_auc\": \"0.807893301740+0.811027480732\", \"selected_public_auc\": \"0.802602534662+0.802275173824\"}, {\"best_public_auc\": 0.8672530039, \"dataset\": \"train_15\", \"final_candidate\": \"rank_top2\", \"final_private_auc\": 0.8711218727, \"oracle_best_private_auc\": 0.8711218727, \"oracle_candidate\": \"rank_top2\", \"selected\": \"rank_top2+rank_top3\", \"selected_private_auc\": \"0.871121872695+0.868312481565\", \"selected_public_auc\": \"0.867253003914+0.865183969074\"}, {\"best_public_auc\": 0.9125669367, \"dataset\": \"train_16\", \"final_candidate\": \"catboost_native\", \"final_private_auc\": 0.9080022004, \"oracle_best_private_auc\": 0.9080022004, \"oracle_candidate\": \"catboost_native\", \"selected\": \"catboost_native+rank_top2\", \"selected_private_auc\": \"0.908002200371+0.897707963191\", \"selected_public_auc\": \"0.912566936694+0.902593059306\"}]")
CANDIDATES = json.loads("[{\"candidate\": \"rank_top2\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7155103808, \"prediction_sha256\": \"3f9f3f0d0522176fe01c198ebccfde46720dcd133ab4fcac937fe7b48a198998\", \"private_auc\": 0.7126573427, \"public_auc\": 0.7203611241}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7134900498, \"prediction_sha256\": \"50e011dfa5f0608a9326d89e947e47e18cc85ab87f89dabc4e47182ebfa45ec8\", \"private_auc\": 0.7094051943, \"public_auc\": 0.7154802316}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7128250696, \"prediction_sha256\": \"e2cac0373552094234ccb04e44c164856d6f9fb8a545c0dd59f9c11222cdcf69\", \"private_auc\": 0.7092094862, \"public_auc\": 0.7165165543}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7153379043, \"prediction_sha256\": \"56736eed5ddac202b13cb530e870337db7fc73d91fb1d1cb336c362ca1f20f8a\", \"private_auc\": 0.7119690835, \"public_auc\": 0.7194964019}, {\"candidate\": \"logistic\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7132097532, \"prediction_sha256\": \"ede19e72c05018ad6a6eb06060e99e164701d150143ad6e3c4ad46d7bd9de02c\", \"private_auc\": 0.7124083468, \"public_auc\": 0.7216405674}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.7076202722, \"prediction_sha256\": \"1fb98de4d9c711be81143299e5515fccf010ed7249e9c2bf814d3dfbc23c321a\", \"private_auc\": 0.7101685043, \"public_auc\": 0.7166664747}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_01\", \"manifest_sha256\": \"2061edfdecdc667fc9c780d322ffb30c73335333648b65f4f77a143a21009886\", \"oof_auc\": 0.6968264586, \"prediction_sha256\": \"cb9cc05bc1a5df1d29322ac6e4bb0108ee6327bb23807726547bcc4dd7eeadc2\", \"private_auc\": 0.6969562017, \"public_auc\": 0.6988574691}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9681423928, \"prediction_sha256\": \"dbe656f5ea5fa27cba3a37355696241fe90114ec2f26aa9e922e4844368cf301\", \"private_auc\": 0.9689826549, \"public_auc\": 0.9704164384}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9675390437, \"prediction_sha256\": \"5a5ccf239ae6851cfae5e4392748178be45b3dda9c0819844704ff5953fc8cbf\", \"private_auc\": 0.9689445725, \"public_auc\": 0.9700735505}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9678385241, \"prediction_sha256\": \"e1517271c02d02578aaed1895cd32b8c9bc66adc494e2d547796a38d8890e020\", \"private_auc\": 0.968844406, \"public_auc\": 0.9705476414}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9673418221, \"prediction_sha256\": \"e26dbcc0abefc01e4161d6115fae9e75cc0a72b46f47eebcd8217b3141412e45\", \"private_auc\": 0.9681915643, \"public_auc\": 0.9691944102}, {\"candidate\": \"logistic\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.7900530335, \"prediction_sha256\": \"370f27bafc2e9d75c84a9f9d458d6d95088dd4c7c60841210c78ad82b59d36f9\", \"private_auc\": 0.7825131208, \"public_auc\": 0.7855204984}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9532083887, \"prediction_sha256\": \"238c511b6beb46aa4f9b457492967b43a214248f2f1e4fc46dbc6faad7b22402\", \"private_auc\": 0.9563970494, \"public_auc\": 0.9567327631}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_02\", \"manifest_sha256\": \"a4c44dc60619e5046dbdbb31637245850fa28f2edba2a405bf0112059fa64df4\", \"oof_auc\": 0.9438286544, \"prediction_sha256\": \"41f5e909104d8ad5e4a77265fb3c7c3db045f8f1c9aca7745fa776a937b24a39\", \"private_auc\": 0.9448489503, \"public_auc\": 0.9445430423}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.8111590553, \"prediction_sha256\": \"3d615018f43a5104bb1dc947ee2b4606fe9c31fe41addc25197247f9237d0f81\", \"private_auc\": 0.8213375559, \"public_auc\": 0.8071077536}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.809483619, \"prediction_sha256\": \"e62d7d07c014f4f97f6369a7112595a244fe244b8cb01262f6f04e193ce9bfe3\", \"private_auc\": 0.8218244085, \"public_auc\": 0.8070526289}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.8108370883, \"prediction_sha256\": \"dbcd8ee658ffcc764080659e49873aef80c68a920ae9ae7777fb9bc380b91d2d\", \"private_auc\": 0.8203130451, \"public_auc\": 0.8066780372}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.8154316056, \"prediction_sha256\": \"5cee55686434cf298a9b523e987c421ed14ec31a8ef984a6d2495827e07e3e9d\", \"private_auc\": 0.8230879052, \"public_auc\": 0.8102336182}, {\"candidate\": \"logistic\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.8058504337, \"prediction_sha256\": \"56b0961e283d737510461a6b675d7b6cd01b7da13201cd1c11201060a76e6c19\", \"private_auc\": 0.8091041527, \"public_auc\": 0.8000462759}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.8004825753, \"prediction_sha256\": \"1fecddea86d8f6619a23614426f0d1bcde04f69ec8b25a8815d0a5a2a3f2a33d\", \"private_auc\": 0.8182910664, \"public_auc\": 0.8044572093}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_03\", \"manifest_sha256\": \"4d1720d53b3a51f662672890cf356ef15b900a5cca2d6ab46cadd5c013f31ab6\", \"oof_auc\": 0.7871704692, \"prediction_sha256\": \"ce64bc346f07ed0c607bcf8035e58a567f26a95b2491364700a9079d07c2f15e\", \"private_auc\": 0.8015754984, \"public_auc\": 0.7913601007}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.8289733438, \"prediction_sha256\": \"e9935df1be4669807187983a90c6f65a92e3487f36421ac6a4faea5349676b32\", \"private_auc\": 0.8367116712, \"public_auc\": 0.8347938228}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.8274965161, \"prediction_sha256\": \"6c3b8bc499aca3d154e154a4975cb60e7982b97cef06af4434b2746ca0a4be2b\", \"private_auc\": 0.8331205921, \"public_auc\": 0.8334649873}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.8282877869, \"prediction_sha256\": \"d7bdd9973033c789b9c769e89eb29e3252bf3e352051a4bd84c2874e6e97ac95\", \"private_auc\": 0.8360216022, \"public_auc\": 0.8333184769}, {\"candidate\": \"logistic\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.7469188385, \"prediction_sha256\": \"c6f19bffac5e04f3ff42946bdd744dd8bbed4d4a54d248c6a9dd31108166f8ee\", \"private_auc\": 0.7531381938, \"public_auc\": 0.743851365}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.8226874886, \"prediction_sha256\": \"c95caa2bbe29c789094b5f578d19ffea1d02473acbba6ae905eb001a9d4cf536\", \"private_auc\": 0.8354053005, \"public_auc\": 0.8314829763}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_04\", \"manifest_sha256\": \"5283b6b23e19601b4f35043ef464162536c008f087afeedbfeeea0dd09dc08c3\", \"oof_auc\": 0.8163553479, \"prediction_sha256\": \"e83aa2457a5cf281d53fed690f1314b558449f37f29b273175afb596197aa736\", \"private_auc\": 0.8265367337, \"public_auc\": 0.8221170342}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.6863820212, \"prediction_sha256\": \"9247941b0da30cb2bdd6e89c0e222eafd7052939aa8696c5b619e11abae18fb1\", \"private_auc\": 0.6868203916, \"public_auc\": 0.6831279973}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.681475732, \"prediction_sha256\": \"68f8e4d191684d5ed59e35f7eef1f201de27c906ecdb9c065dd27ab9776c7dc5\", \"private_auc\": 0.6791694585, \"public_auc\": 0.6749442742}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.6824655349, \"prediction_sha256\": \"317e877b86c6efc5ec57eb8bb8014f610c8c27ab5feb4952ce2bb03a134ca50a\", \"private_auc\": 0.6853103685, \"public_auc\": 0.6805875913}, {\"candidate\": \"logistic\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.6734682978, \"prediction_sha256\": \"90b06f4573cb177f129719a324c80645e0b6ab7284ed3d255d0022ef488213dc\", \"private_auc\": 0.6814080085, \"public_auc\": 0.6789708893}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.6530349208, \"prediction_sha256\": \"9d8b00bf4752186cdd8dda3fe4ca91428db63aba6069ed4eabc8eb176523cc7b\", \"private_auc\": 0.6723217053, \"public_auc\": 0.6666187632}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_05\", \"manifest_sha256\": \"e5657f673268c69281548e6d36cd3160143dcb1acc48ca43ff03453f9691bee9\", \"oof_auc\": 0.6486235331, \"prediction_sha256\": \"dc6b8df5035aaf84629eaa167f089ede8ccd81619009b1a5555aec3aa21f3062\", \"private_auc\": 0.6422415042, \"public_auc\": 0.6446160905}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.8117528037, \"prediction_sha256\": \"536f262120cc4e72e092777993b1a1fb5b696a6ea57d23a407863dbe0ebda6b0\", \"private_auc\": 0.8154932656, \"public_auc\": 0.8019570083}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.8107075952, \"prediction_sha256\": \"727ede701f3648aaa12b87311a6d077534850290ed2b0bde9d24acb053a33b45\", \"private_auc\": 0.8152197657, \"public_auc\": 0.8011714882}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.8108995432, \"prediction_sha256\": \"b54b43e613c48c479f8d6309d6438f3540714047b4c0876a7b73f19adac5dc2e\", \"private_auc\": 0.8147808696, \"public_auc\": 0.8011946082}, {\"candidate\": \"logistic\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.8071456908, \"prediction_sha256\": \"ec271079c1f466de98fed3a271f37b14a08546441dfadff82b4d34960c140aca\", \"private_auc\": 0.8120060218, \"public_auc\": 0.7982420477}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.8109214459, \"prediction_sha256\": \"3e11d0ef579a60e42e13289456c722b4d8845e411835f0262803597752384314\", \"private_auc\": 0.8148064752, \"public_auc\": 0.8020350083}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_06\", \"manifest_sha256\": \"1e95be5edfc97f3ad10ba3d0479e7d203a211da4fe5529314ce051613d904de1\", \"oof_auc\": 0.7681274872, \"prediction_sha256\": \"879391308d090607c205c503cfed24cbd4a3fc48e40d74a01bbc4eda2370bfd8\", \"private_auc\": 0.787933709, \"public_auc\": 0.775168604}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.8245907904, \"prediction_sha256\": \"0fbaa42a51b327b1680421283a6f4e56d403387b98268e47862d399c21fad9d1\", \"private_auc\": 0.8328678714, \"public_auc\": 0.8332486767}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.8189817398, \"prediction_sha256\": \"aa20ebbd323b8c0edb18113de4419f4d421deefb6028465ae92e310f9236b573\", \"private_auc\": 0.8297734336, \"public_auc\": 0.8290828202}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.81811789, \"prediction_sha256\": \"80bf93da90c350e845eb0a613ce675b6f07c8b9182b7ddf278aa1efde55096bb\", \"private_auc\": 0.829259401, \"public_auc\": 0.8261969746}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.8243419404, \"prediction_sha256\": \"c27dfc2f6233c1e336f30b14a35232612ae9d4518f9b74bd499eea754f5391c3\", \"private_auc\": 0.8328390651, \"public_auc\": 0.8321353385}, {\"candidate\": \"logistic\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.7054837537, \"prediction_sha256\": \"0886ba634cff355f28706ab96e6e95899cc72fcead152471c0f799c0d77cb85e\", \"private_auc\": 0.7130102578, \"public_auc\": 0.7086153686}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.8205376279, \"prediction_sha256\": \"b2695159b698cb8748515cf4ca98dc0f17d7cc68449de32a71419a46f36fa8c3\", \"private_auc\": 0.8301796025, \"public_auc\": 0.8312939736}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_07\", \"manifest_sha256\": \"be6fc95a68b0a9e1e8b3e8c58591782f3f058fe37e521b963f50231ea4d37df9\", \"oof_auc\": 0.7983127978, \"prediction_sha256\": \"81003d9fd2c143a17a4b7cd272a1dde2c8f2507f5954a3344c1c3e666788eea4\", \"private_auc\": 0.8098828367, \"public_auc\": 0.8123550616}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.8548840989, \"prediction_sha256\": \"03735b2c8878310a4cd36c3e5c9a8789c3808cfadb931648028e2d195c904e57\", \"private_auc\": 0.8502486081, \"public_auc\": 0.8546659413}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.851875287, \"prediction_sha256\": \"f2c156947ae434f8500066cd9cc9b50d958fa5130336cb74c2b66a42551383df\", \"private_auc\": 0.8488513435, \"public_auc\": 0.8522521536}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.8540996564, \"prediction_sha256\": \"31fdba1491d0b9f95bec62254ef74022fee07bba191c881317c0573ca5d13a88\", \"private_auc\": 0.8496311507, \"public_auc\": 0.8544233062}, {\"candidate\": \"logistic\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.8542216049, \"prediction_sha256\": \"0f119d226b1b8dbbe52d6aab28e4e0f9b99579845b013b0ef9bbf9cd61b633af\", \"private_auc\": 0.8498087907, \"public_auc\": 0.8548148675}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.8493649032, \"prediction_sha256\": \"bd437b09fc190eda1216262946969b1d6f0ac51bd80d2babbcd37053f24713a2\", \"private_auc\": 0.8465933371, \"public_auc\": 0.8520101587}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_08\", \"manifest_sha256\": \"ac079609676a18fdb22f1bb38f7f9a7f931e8ba891b7321a92ee35de15d00ddf\", \"oof_auc\": 0.830875072, \"prediction_sha256\": \"7fabfbae4143925e98d470bcf5b42e8ca13cc0d379b27a6441a5a563408b0228\", \"private_auc\": 0.8288914506, \"public_auc\": 0.8306582647}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6575090741, \"prediction_sha256\": \"04de20fa54a701f5e522f0b32e7e409a7d82b2859e4b82de0c7ee5c714e4db3c\", \"private_auc\": 0.6533474602, \"public_auc\": 0.6398983908}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6439954727, \"prediction_sha256\": \"43748198b47d38d21d89a1a6d7b57554f9cfaabda51e553a67be2661068cb697\", \"private_auc\": 0.65065167, \"public_auc\": 0.6349798745}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6367687044, \"prediction_sha256\": \"e23960b7b9e15dc1a532ec5bb17c14c92508a30a5ca244388937198a165a6c4e\", \"private_auc\": 0.6520810508, \"public_auc\": 0.6351115748}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6548437561, \"prediction_sha256\": \"2211a26586ff26875490c4088ecd4f730f246570e0c2a8b9275cd74c4ead897d\", \"private_auc\": 0.6554791306, \"public_auc\": 0.640452476}, {\"candidate\": \"logistic\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6459046145, \"prediction_sha256\": \"7faae8393c57bed3d4f495f684bb7e2f46f917a33214340afe8679784cda30db\", \"private_auc\": 0.6409889082, \"public_auc\": 0.6304814228}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6121807798, \"prediction_sha256\": \"5bac2caedfd6669947899425a7720fe68534e8d68a94f45553d2fec33fefd8a2\", \"private_auc\": 0.6464233716, \"public_auc\": 0.6327422504}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_09\", \"manifest_sha256\": \"56af7fd8064555c1491c4b2bce59f91aedaa3a8b26f8793965d7f7e63706a829\", \"oof_auc\": 0.6116896718, \"prediction_sha256\": \"1b757e88aa124b082c3b06aadb662ee77f9c3c86586c92a5114964b3befa49dc\", \"private_auc\": 0.6394956828, \"public_auc\": 0.6292929201}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.8484245558, \"prediction_sha256\": \"6c15cd281955a48023714beb0f76b29a016e7c8494d40107589cc5e5c4ce9708\", \"private_auc\": 0.8574972248, \"public_auc\": 0.8466805355}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.8472450612, \"prediction_sha256\": \"1ceecf1b66bd56ad158eccc3752b76dad3d0cef3d3523f9ff41a800afa938dfd\", \"private_auc\": 0.8564033976, \"public_auc\": 0.8457162953}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.847233455, \"prediction_sha256\": \"daf4861b2d5bfd225893c7fc11fa31047980a7a7deb92b363dfd3a60a29cd37b\", \"private_auc\": 0.8548806327, \"public_auc\": 0.843725975}, {\"candidate\": \"logistic\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.5998871958, \"prediction_sha256\": \"dd89b118be54a2cf542f0b88d8eebd9a41d5b9d6c6593c9b344a4e2986598e4e\", \"private_auc\": 0.5999316068, \"public_auc\": 0.6021154563}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.8382444229, \"prediction_sha256\": \"4a6735d02b69d85463a9ed0d1c80da9b5981106e76bdf229653ef1d090bab4b2\", \"private_auc\": 0.8519307958, \"public_auc\": 0.8413374146}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_10\", \"manifest_sha256\": \"ee86a85325a4544625f58f63d17d1b07dd2769cbf145986c61b7c08e80e262f8\", \"oof_auc\": 0.8274524364, \"prediction_sha256\": \"4c55b98fc60af10692cfb7ad6bc35f94d894ea3d6a61dea6b4bcc6474e564f77\", \"private_auc\": 0.8354362563, \"public_auc\": 0.8243250919}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.8235382347, \"prediction_sha256\": \"2f68f29c6384e14262a855151ff772bf3849b7976166527befc19e1d645169f6\", \"private_auc\": 0.8231533122, \"public_auc\": 0.8357659503}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.8184419736, \"prediction_sha256\": \"6ec45615a09c8a1a3ee63541cbf29c144f4d0e9c9d2f4fda534935a8e0b25115\", \"private_auc\": 0.8194348652, \"public_auc\": 0.832090541}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.8210086553, \"prediction_sha256\": \"838f04eacc639092954eaecf4413c03998b382cef0031300f7397337c6051ff1\", \"private_auc\": 0.8211695321, \"public_auc\": 0.8331091677}, {\"candidate\": \"logistic\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.6707188761, \"prediction_sha256\": \"9b7c91f446f13f295f296120adf7eb41ed65945b9f5c13d3c661d29da5d12dde\", \"private_auc\": 0.6702656802, \"public_auc\": 0.6777099102}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.804829022, \"prediction_sha256\": \"8cd8d7f98887300b74dd06fb8ce401bac115af48020db1ec6d699942409ab1fc\", \"private_auc\": 0.8099531936, \"public_auc\": 0.8206351315}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_11\", \"manifest_sha256\": \"9d9b5594fdb6a7e34e51a4c2082e18c8ab1668f2b7fa78b9508efb63603974de\", \"oof_auc\": 0.8180055514, \"prediction_sha256\": \"164be0e9222162ada79d80d94640cac4ba67049eb045c337ca4a9919fa9ed0a1\", \"private_auc\": 0.8186130798, \"public_auc\": 0.8303811749}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.787497671, \"prediction_sha256\": \"bfaa5d3f461b0d8a0ae3145faf49f5564eccf3859ed8b55d29994b5ff9ed6a73\", \"private_auc\": 0.7728086984, \"public_auc\": 0.7948010645}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.7875589523, \"prediction_sha256\": \"ec8f48cc13323f900058c6c5a81f3399c9d2139cd9e9e0c001bc96e84b8329a8\", \"private_auc\": 0.7727979784, \"public_auc\": 0.7950334648}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.7869720477, \"prediction_sha256\": \"8c56c54087b398e616ba2a8cbec5db14d9a934e2e93a0184ce0a5f090e7a987d\", \"private_auc\": 0.7725686978, \"public_auc\": 0.7943553839}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.7874252696, \"prediction_sha256\": \"c421308b32288bf3f205370b0f0432047442c39b8d2451be5177f9915cc0ea33\", \"private_auc\": 0.7723861373, \"public_auc\": 0.7952189851}, {\"candidate\": \"logistic\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.6111712359, \"prediction_sha256\": \"0afe8788da8c8a66cde59c5a242c1f0022b93494a59bc55275439a610ba22e66\", \"private_auc\": 0.5967024876, \"public_auc\": 0.6082725559}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.785677276, \"prediction_sha256\": \"6790fd903273d11665396f917dcbd891dfab3dd29cdbde330324cb55f6f532b1\", \"private_auc\": 0.7706186928, \"public_auc\": 0.7953025852}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_12\", \"manifest_sha256\": \"6aa87fc582feb2e5c2d7778b7332c60e1dcf567aeaec66ed89955cd58ffe106e\", \"oof_auc\": 0.7612132409, \"prediction_sha256\": \"002744419818c449b792d2a9aa4b158683572efab95a757cef6e8c0cae2a89da\", \"private_auc\": 0.7560451355, \"public_auc\": 0.7740318346}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.676608, \"prediction_sha256\": \"8e5b202ed6f74b1a409f6f3d75083d6c5278cacf334452b87988d16ab63b0f37\", \"private_auc\": 0.6486488346, \"public_auc\": 0.6366350265}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.674528, \"prediction_sha256\": \"5011888ed75823b49cb7ab604e9deb010eba76067ed50ca5236e4e868560b5ad\", \"private_auc\": 0.6488712355, \"public_auc\": 0.6397187189}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.668544, \"prediction_sha256\": \"3dce6b9c6f5a1f2da73db450958114e90e6a67e7d0d6d1c04523ede84589f17c\", \"private_auc\": 0.6508653235, \"public_auc\": 0.6377303109}, {\"candidate\": \"logistic\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.65936, \"prediction_sha256\": \"2ef90f348da3033092f3bcdf609384cc0918f764a51391ae805d86a4f2d13252\", \"private_auc\": 0.6414036856, \"public_auc\": 0.6278740315}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.645056, \"prediction_sha256\": \"e98525622a51bccf64c6e59eda7667ced414e754cc5eef1d5f9472c8e8550f49\", \"private_auc\": 0.6511157245, \"public_auc\": 0.6364023856}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_13\", \"manifest_sha256\": \"203897bad6c6d982055a8cd779537a6bd897d9a965d54bcec80bce6a91ce36d4\", \"oof_auc\": 0.62744, \"prediction_sha256\": \"9df43a8a0592f16805bdb607336f04e61c3bc079f478de65b0578f93d5aabe28\", \"private_auc\": 0.631006524, \"public_auc\": 0.6076256305}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.8073168, \"prediction_sha256\": \"201f637b59b45137299f59869ad9c953a1ee298b49127c8afa8cae4bdba06b43\", \"private_auc\": 0.8103099631, \"public_auc\": 0.7996334871}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.8060899121, \"prediction_sha256\": \"e085cd5881a9663927e925e318858cf9447caa5a48452106cae4562853717a8f\", \"private_auc\": 0.8114780095, \"public_auc\": 0.7992607661}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.8058429622, \"prediction_sha256\": \"cff21590662e4da59071cd49ba64327e7e15603ddf91b098a9944d79471aba33\", \"private_auc\": 0.8079659496, \"public_auc\": 0.7989318053}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.8090946932, \"prediction_sha256\": \"59acaa32f0b9dca531cc24331b41d01a0b75c992de856bd89cd24e1e808b29f4\", \"private_auc\": 0.8110274807, \"public_auc\": 0.8022751738}, {\"candidate\": \"logistic\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.6798852635, \"prediction_sha256\": \"8dd624d553c25380dadf44a3138811c0b08d46579ac933bf0d5567fcf5320a76\", \"private_auc\": 0.6746742528, \"public_auc\": 0.680474382}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.8046681389, \"prediction_sha256\": \"a5033a65003e72a5529d3e088e0df6f0788ea382649b2e6838e84cf2c8147bc7\", \"private_auc\": 0.8078933017, \"public_auc\": 0.8026025347}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_14\", \"manifest_sha256\": \"69fab5370518afef6b8c1b544f4d6d6e11f959694056b63fbe52d024d32a3e91\", \"oof_auc\": 0.7810282516, \"prediction_sha256\": \"9c8b8f9c2d3be51c15d39f79b740b28333a89e95fb3edebcc75f0c5753aec0a5\", \"private_auc\": 0.7914408022, \"public_auc\": 0.7843210479}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.884024, \"prediction_sha256\": \"7052e24fff3e06d827caa416eee4f34a865e9d22c8d4731360cae6d8132f7a3a\", \"private_auc\": 0.8711218727, \"public_auc\": 0.8672530039}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.867248, \"prediction_sha256\": \"ea4297b276f3d1b0f8f547a26b03bbc20af7a3135d2a8ce34e542571377ce973\", \"private_auc\": 0.8572435269, \"public_auc\": 0.8528980493}, {\"candidate\": \"catboost_ordinal\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.84332, \"prediction_sha256\": \"ff2f6b1cffb7d4bebc8d1c009c2e95a64aa55bd9a2c79a1cb4eb0bb8bc4de0b0\", \"private_auc\": 0.8478359438, \"public_auc\": 0.8428420262}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.883064, \"prediction_sha256\": \"5e71f23dcd582c7221cf44207d055414a0643078bfce1551ef4ef948bc8aff00\", \"private_auc\": 0.8683124816, \"public_auc\": 0.8651839691}, {\"candidate\": \"logistic\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.869104, \"prediction_sha256\": \"1b7f2b5495b5e9cbbc302e7cb0576d708252704d0f8f2536cbd81d2978ed2915\", \"private_auc\": 0.8635713189, \"public_auc\": 0.8601121807}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.854176, \"prediction_sha256\": \"3eb4be5b2b75afcb26ad103bff820a42b45a355ca487d6e8cfb5f03f2511b494\", \"private_auc\": 0.8493619953, \"public_auc\": 0.8471199896}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_15\", \"manifest_sha256\": \"71507e1fc0d0c558a9237e98043dcef029960d0f3f57e553b582550b4863db69\", \"oof_auc\": 0.793976, \"prediction_sha256\": \"f24cbbcd87ca2435e6f315276ae806c6136952f92cc4f4694feef00439723763\", \"private_auc\": 0.7864380198, \"public_auc\": 0.7806135605}, {\"candidate\": \"rank_top2\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.8986082624, \"prediction_sha256\": \"4f48d9c1e8b5f688721b894a6af92826ff97ffb30f44c4ebd81697b075d7dcdc\", \"private_auc\": 0.8977079632, \"public_auc\": 0.9025930593}, {\"candidate\": \"catboost_native\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.9064555795, \"prediction_sha256\": \"ee05bd3635c3194e9a51d006c770cdfba9ab6f0399a6923f8361f6c9fba596c8\", \"private_auc\": 0.9080022004, \"public_auc\": 0.9125669367}, {\"candidate\": \"rank_top3\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.8948318938, \"prediction_sha256\": \"5e2c97625788a7bbbec39f6508ed022f108336e632c05c4ab4d22cfaf8013bf7\", \"private_auc\": 0.8975029185, \"public_auc\": 0.901669527}, {\"candidate\": \"logistic\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.8009996162, \"prediction_sha256\": \"a941e3b4190577cfc582cd409d0e1e72bb84271601b638defec5d1a23ec2df32\", \"private_auc\": 0.7942456594, \"public_auc\": 0.7997830183}, {\"candidate\": \"hist_gradient_boosting\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.8755356833, \"prediction_sha256\": \"068d1d316af47e24107575096ac0eb7bc30be2702aebc6cee0235b527dc92406\", \"private_auc\": 0.891622943, \"public_auc\": 0.8940428443}, {\"candidate\": \"extra_trees\", \"dataset\": \"train_16\", \"manifest_sha256\": \"7e191d10aed182890398afc98a44b89b025c2d68330b5b3a87987b8aea174c8d\", \"oof_auc\": 0.8768973517, \"prediction_sha256\": \"727f2908d7d524f05598db3f185a35c895eb66cc6a8ca1614cd913b2fd5e76aa\", \"private_auc\": 0.8770134864, \"public_auc\": 0.8815708771}]")
PROBE = json.loads("[{\"augmented_final_private_auc\": 0.7094051943, \"augmented_selected\": \"schema_probe+rank_top2\", \"baseline_final_private_auc\": 0.7050047207, \"baseline_selected\": \"rank_top2+rank_all\", \"dataset\": \"train_01\", \"final_delta\": 0.0044004737, \"probe_private_auc\": 0.7094051943, \"probe_public_auc\": 0.7154802316, \"probe_runtime_seconds\": 20.092637167, \"probe_sha256\": \"50e011dfa5f0608a9326d89e947e47e18cc85ab87f89dabc4e47182ebfa45ec8\"}, {\"augmented_final_private_auc\": 0.9689445725, \"augmented_selected\": \"schema_probe+rank_top2\", \"baseline_final_private_auc\": 0.968359975, \"baseline_selected\": \"rank_top2+catboost\", \"dataset\": \"train_02\", \"final_delta\": 0.0005845974, \"probe_private_auc\": 0.9689445725, \"probe_public_auc\": 0.9700735505, \"probe_runtime_seconds\": 63.421727, \"probe_sha256\": \"5a5ccf239ae6851cfae5e4392748178be45b3dda9c0819844704ff5953fc8cbf\"}, {\"augmented_final_private_auc\": 0.8218244085, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.7905067812, \"baseline_selected\": \"catboost+rank_all\", \"dataset\": \"train_03\", \"final_delta\": 0.0313176273, \"probe_private_auc\": 0.8218244085, \"probe_public_auc\": 0.8070526289, \"probe_runtime_seconds\": 12.088200917, \"probe_sha256\": \"e62d7d07c014f4f97f6369a7112595a244fe244b8cb01262f6f04e193ce9bfe3\"}, {\"augmented_final_private_auc\": 0.8358894289, \"augmented_selected\": \"rank_top2+catboost\", \"baseline_final_private_auc\": 0.8358894289, \"baseline_selected\": \"rank_top2+catboost\", \"dataset\": \"train_04\", \"final_delta\": 0.0, \"probe_private_auc\": 0.8331205921, \"probe_public_auc\": 0.8334649873, \"probe_runtime_seconds\": 11.366252209, \"probe_sha256\": \"6c3b8bc499aca3d154e154a4975cb60e7982b97cef06af4434b2746ca0a4be2b\"}, {\"augmented_final_private_auc\": 0.6791694585, \"augmented_selected\": \"schema_probe+rank_all\", \"baseline_final_private_auc\": 0.5568932731, \"baseline_selected\": \"rank_all+rank_top2\", \"dataset\": \"train_05\", \"final_delta\": 0.1222761854, \"probe_private_auc\": 0.6791694585, \"probe_public_auc\": 0.6749442742, \"probe_runtime_seconds\": 5.6757405, \"probe_sha256\": \"68f8e4d191684d5ed59e35f7eef1f201de27c906ecdb9c065dd27ab9776c7dc5\"}, {\"augmented_final_private_auc\": 0.8152197657, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.5, \"baseline_selected\": \"catboost+lightgbm\", \"dataset\": \"train_06\", \"final_delta\": 0.3152197657, \"probe_private_auc\": 0.8152197657, \"probe_public_auc\": 0.8011714882, \"probe_runtime_seconds\": 18.505835833, \"probe_sha256\": \"727ede701f3648aaa12b87311a6d077534850290ed2b0bde9d24acb053a33b45\"}, {\"augmented_final_private_auc\": 0.8297734336, \"augmented_selected\": \"schema_probe+rank_all\", \"baseline_final_private_auc\": 0.7562754946, \"baseline_selected\": \"rank_all+rank_top2\", \"dataset\": \"train_07\", \"final_delta\": 0.073497939, \"probe_private_auc\": 0.8297734336, \"probe_public_auc\": 0.8290828202, \"probe_runtime_seconds\": 30.381004708, \"probe_sha256\": \"aa20ebbd323b8c0edb18113de4419f4d421deefb6028465ae92e310f9236b573\"}, {\"augmented_final_private_auc\": 0.8488513435, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.5, \"baseline_selected\": \"catboost+lightgbm\", \"dataset\": \"train_08\", \"final_delta\": 0.3488513435, \"probe_private_auc\": 0.8488513435, \"probe_public_auc\": 0.8522521536, \"probe_runtime_seconds\": 15.619488625, \"probe_sha256\": \"f2c156947ae434f8500066cd9cc9b50d958fa5130336cb74c2b66a42551383df\"}, {\"augmented_final_private_auc\": 0.6527382172, \"augmented_selected\": \"schema_probe+rank_all\", \"baseline_final_private_auc\": 0.6528638261, \"baseline_selected\": \"rank_all+catboost\", \"dataset\": \"train_09\", \"final_delta\": -0.0001256089, \"probe_private_auc\": 0.65065167, \"probe_public_auc\": 0.6349798745, \"probe_runtime_seconds\": 4.880470875, \"probe_sha256\": \"43748198b47d38d21d89a1a6d7b57554f9cfaabda51e553a67be2661068cb697\"}, {\"augmented_final_private_auc\": 0.8564033976, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.857216387, \"baseline_selected\": \"catboost+rank_top2\", \"dataset\": \"train_10\", \"final_delta\": -0.0008129894, \"probe_private_auc\": 0.8564033976, \"probe_public_auc\": 0.8457162953, \"probe_runtime_seconds\": 15.526368542, \"probe_sha256\": \"1ceecf1b66bd56ad158eccc3752b76dad3d0cef3d3523f9ff41a800afa938dfd\"}, {\"augmented_final_private_auc\": 0.8196257026, \"augmented_selected\": \"rank_top2+schema_probe\", \"baseline_final_private_auc\": 0.8196257026, \"baseline_selected\": \"rank_top2+rank_all\", \"dataset\": \"train_11\", \"final_delta\": 0.0, \"probe_private_auc\": 0.8194348652, \"probe_public_auc\": 0.832090541, \"probe_runtime_seconds\": 43.388931583, \"probe_sha256\": \"6ec45615a09c8a1a3ee63541cbf29c144f4d0e9c9d2f4fda534935a8e0b25115\"}, {\"augmented_final_private_auc\": 0.7727979784, \"augmented_selected\": \"schema_probe+lightgbm\", \"baseline_final_private_auc\": 0.7532521683, \"baseline_selected\": \"lightgbm+rank_all\", \"dataset\": \"train_12\", \"final_delta\": 0.01954581, \"probe_private_auc\": 0.7727979784, \"probe_public_auc\": 0.7950334648, \"probe_runtime_seconds\": 52.086106292, \"probe_sha256\": \"ec8f48cc13323f900058c6c5a81f3399c9d2139cd9e9e0c001bc96e84b8329a8\"}, {\"augmented_final_private_auc\": 0.6488712355, \"augmented_selected\": \"schema_probe+extra_trees\", \"baseline_final_private_auc\": 0.4988344753, \"baseline_selected\": \"extra_trees+lightgbm\", \"dataset\": \"train_13\", \"final_delta\": 0.1500367601, \"probe_private_auc\": 0.6488712355, \"probe_public_auc\": 0.6397187189, \"probe_runtime_seconds\": 1.888743125, \"probe_sha256\": \"5011888ed75823b49cb7ab604e9deb010eba76067ed50ca5236e4e868560b5ad\"}, {\"augmented_final_private_auc\": 0.8114780095, \"augmented_selected\": \"schema_probe+rank_top2\", \"baseline_final_private_auc\": 0.7445568513, \"baseline_selected\": \"rank_top2+catboost\", \"dataset\": \"train_14\", \"final_delta\": 0.0669211582, \"probe_private_auc\": 0.8114780095, \"probe_public_auc\": 0.7992607661, \"probe_runtime_seconds\": 29.438689625, \"probe_sha256\": \"e085cd5881a9663927e925e318858cf9447caa5a48452106cae4562853717a8f\"}, {\"augmented_final_private_auc\": 0.8572435269, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.6322377612, \"baseline_selected\": \"catboost+rank_all\", \"dataset\": \"train_15\", \"final_delta\": 0.2250057657, \"probe_private_auc\": 0.8572435269, \"probe_public_auc\": 0.8528980493, \"probe_runtime_seconds\": 10.7842375, \"probe_sha256\": \"ea4297b276f3d1b0f8f547a26b03bbc20af7a3135d2a8ce34e542571377ce973\"}, {\"augmented_final_private_auc\": 0.9080022004, \"augmented_selected\": \"schema_probe+catboost\", \"baseline_final_private_auc\": 0.9071317803, \"baseline_selected\": \"catboost+rank_top2\", \"dataset\": \"train_16\", \"final_delta\": 0.0008704201, \"probe_private_auc\": 0.9080022004, \"probe_public_auc\": 0.9125669367, \"probe_runtime_seconds\": 6.393304583, \"probe_sha256\": \"ee05bd3635c3194e9a51d006c770cdfba9ab6f0399a6923f8361f6c9fba596c8\"}]")
SCHEMAS = json.loads("[{\"categorical\": 5, \"dataset\": \"train_01\", \"features\": 12, \"missing_columns\": 5, \"numeric\": 7, \"test_rows\": 10000, \"train_rows\": 14957}, {\"categorical\": 2, \"dataset\": \"train_02\", \"features\": 28, \"missing_columns\": 16, \"numeric\": 26, \"test_rows\": 10000, \"train_rows\": 14929}, {\"categorical\": 7, \"dataset\": \"train_03\", \"features\": 18, \"missing_columns\": 9, \"numeric\": 11, \"test_rows\": 10000, \"train_rows\": 3501}, {\"categorical\": 0, \"dataset\": \"train_04\", \"features\": 12, \"missing_columns\": 9, \"numeric\": 12, \"test_rows\": 10000, \"train_rows\": 8775}, {\"categorical\": 5, \"dataset\": \"train_05\", \"features\": 9, \"missing_columns\": 5, \"numeric\": 4, \"test_rows\": 10000, \"train_rows\": 1060}, {\"categorical\": 9, \"dataset\": \"train_06\", \"features\": 9, \"missing_columns\": 5, \"numeric\": 0, \"test_rows\": 10000, \"train_rows\": 10803}, {\"categorical\": 6, \"dataset\": \"train_07\", \"features\": 17, \"missing_columns\": 15, \"numeric\": 11, \"test_rows\": 10000, \"train_rows\": 10417}, {\"categorical\": 12, \"dataset\": \"train_08\", \"features\": 12, \"missing_columns\": 4, \"numeric\": 0, \"test_rows\": 10000, \"train_rows\": 8173}, {\"categorical\": 4, \"dataset\": \"train_09\", \"features\": 18, \"missing_columns\": 13, \"numeric\": 14, \"test_rows\": 10000, \"train_rows\": 1109}, {\"categorical\": 0, \"dataset\": \"train_10\", \"features\": 27, \"missing_columns\": 20, \"numeric\": 27, \"test_rows\": 10000, \"train_rows\": 11800}, {\"categorical\": 0, \"dataset\": \"train_11\", \"features\": 20, \"missing_columns\": 10, \"numeric\": 20, \"test_rows\": 10000, \"train_rows\": 28879}, {\"categorical\": 3, \"dataset\": \"train_12\", \"features\": 8, \"missing_columns\": 2, \"numeric\": 5, \"test_rows\": 10000, \"train_rows\": 49432}, {\"categorical\": 4, \"dataset\": \"train_13\", \"features\": 9, \"missing_columns\": 8, \"numeric\": 5, \"test_rows\": 10000, \"train_rows\": 500}, {\"categorical\": 15, \"dataset\": \"train_14\", \"features\": 23, \"missing_columns\": 17, \"numeric\": 8, \"test_rows\": 10000, \"train_rows\": 11108}, {\"categorical\": 24, \"dataset\": \"train_15\", \"features\": 30, \"missing_columns\": 17, \"numeric\": 6, \"test_rows\": 10000, \"train_rows\": 500}, {\"categorical\": 0, \"dataset\": \"train_16\", \"features\": 21, \"missing_columns\": 3, \"numeric\": 21, \"test_rows\": 10000, \"train_rows\": 1809}]")
ECOSYSTEM = json.loads("[{\"notebook\": \"Demo Agent\", \"owner\": \"Ryan Holbrook\", \"score_rank\": 19, \"vote_rank\": 1, \"votes\": 92}, {\"notebook\": \"AgentForge\", \"owner\": \"Lucifer\", \"score_rank\": 2, \"vote_rank\": 2, \"votes\": 77}, {\"notebook\": \"AIDE Agent\", \"owner\": \"Ryan Holbrook\", \"score_rank\": 11, \"vote_rank\": 3, \"votes\": 35}, {\"notebook\": \"Meta-learning Pipeline\", \"owner\": \"Avik Das\", \"score_rank\": 9, \"vote_rank\": 5, \"votes\": 25}, {\"notebook\": \"Solution\", \"owner\": \"Kaiwalya\", \"score_rank\": 3, \"vote_rank\": 8, \"votes\": 16}, {\"notebook\": \"Easy Template\", \"owner\": \"Naji Ama\", \"score_rank\": 6, \"vote_rank\": 9, \"votes\": 14}, {\"notebook\": \"Agent Template\", \"owner\": \"Naji Ama\", \"score_rank\": 1, \"vote_rank\": 18, \"votes\": 6}]")
PROVENANCE = json.loads("[{\"license_treatment\": \"competition-supplied; cited\", \"source\": \"Official starter kit\", \"url\": \"https://www.kaggle.com/competitions/autonomous-agent-prediction-beta\", \"use\": \"archive rules, schema, local evaluation protocol\"}, {\"license_treatment\": \"metadata license null; no source reused\", \"source\": \"Ryan Holbrook demo agent\", \"url\": \"https://www.kaggle.com/code/ryanholbrook/autonomous-agent-prediction-beta-demo-agent\", \"use\": \"host-mechanics orientation only\"}, {\"license_treatment\": \"metadata license null; no source reused\", \"source\": \"Naji Ama agent template\", \"url\": \"https://www.kaggle.com/code/najiama/agent-template-lb-0-823-reproducible\", \"use\": \"matched external replay identity only\"}, {\"license_treatment\": \"metadata license null; no source reused\", \"source\": \"Lucifer AgentForge\", \"url\": \"https://www.kaggle.com/code/lucifer19/agentforge-autonomous-binary-classifier\", \"use\": \"portfolio concept attribution only\"}, {\"license_treatment\": \"metadata license null; no source reused\", \"source\": \"Kaiwalya solution\", \"url\": \"https://www.kaggle.com/code/kaiwalyaatulraut/autonomous-agent-prediction-solution\", \"use\": \"robust-tabular concept attribution only\"}]")
RECEIPT = json.loads("{\"agent_solution_access\": false, \"agent_source\": {\"files\": [{\"bytes\": 367, \"path\": \"agent.yaml\", \"sha256\": \"b241986ffc52ff5e5fbbcacba72e5bfbbe48ab63dedb7db383fd3b99add9eec5\"}, {\"bytes\": 107, \"path\": \"configs/sampling.yaml\", \"sha256\": \"5e7668cd9877a77eeb8a84468e92c582523eb168ad165a7f0a44008200e0b6e8\"}, {\"bytes\": 1847, \"path\": \"prompts/system.md\", \"sha256\": \"83300abf7da8d5a4629036c91b345c68387debaef9e5b33eec77a7c96d6456eb\"}, {\"bytes\": 1001, \"path\": \"skills/schema-portfolio/SKILL.md\", \"sha256\": \"3c57c823eaf900ae29a1b92f17617dfc672e379ac21a21ad38a2b93f5833a316\"}, {\"bytes\": 19292, \"path\": \"skills/schema-portfolio/scripts/run_portfolio.py\", \"sha256\": \"8ac114fccb4ad44ebda72dfa98d73d637c493bb917dc48279163f75385a28f0c\"}], \"forbidden_hits\": [], \"forbidden_tokens\": [\"solution.csv\", \"private_auc\", \"public_auc\"]}, \"baseline_mean_final_private_auc\": 0.8039154997448668, \"baseline_selection_sha256\": \"4a30f32ac0876180d29f07e510c256e5163c39a7c39997dcdb90618ee6b2dc36\", \"candidate_rows\": 104, \"candidate_scores_csv\": \"CLEANROOM_CANDIDATE_SCORES.csv\", \"candidate_scores_sha256\": \"88fa1b44574d6c005db45778c2a7566b79fd81d9a9deb77717e46b7296edebfa\", \"created_at_utc\": \"2026-07-29T14:30:47.217949+00:00\", \"datasets\": 16, \"decision\": \"PROMOTE\", \"delta_vs_dtype_repaired_public_baseline\": 0.0008596498041668621, \"mean_best_public_auc\": 0.8031740587967551, \"mean_final_private_auc\": 0.8047751495490336, \"mean_oracle_best_private_auc\": 0.8048453677248177, \"scope\": \"offline replay on 16 official local tasks; not an official Kaggle score\", \"selection_rule\": \"two highest Public AUC candidates, then better Private AUC\", \"selection_scores_csv\": \"CLEANROOM_SELECTION_SCORES.csv\", \"selection_scores_sha256\": \"fc63b6aa233fd8341f09f9196da5eef176cbaa82f517aed99ad4ee4136012df6\", \"tasks_better\": 10, \"tasks_equal\": 0, \"tasks_worse\": 6, \"total_agent_runtime_seconds\": 789.9605571260036}")
SOURCES = json.loads("{\"agent.yaml\": \"name: schema_portfolio_agent\\ndescription: Deterministic schema-aware binary tabular portfolio with public-score selection.\\nmodel: gemini-2.5-flash-lite\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - submit_predictions\\n  - select_submission\\n  - get_status\\nskills:\\n  - skills/schema-portfolio\\ngenerate_content_config: !include configs/sampling.yaml\\n\", \"configs/sampling.yaml\": \"temperature: 0.0\\nmax_output_tokens: 2048\\nthinking_config:\\n  thinking_budget: 256\\n  include_thoughts: false\\n\", \"prompts/system.md\": \"You are a deterministic orchestration agent for a binary tabular prediction task.\\n\\n## Objective\\n\\nMaximize **{metric_name}** ({metric_direction}) for this task:\\n\\n{problem_description}\\n\\n## Hard workflow\\n\\n1. Call `get_status` once and confirm that at least two selections and enough submissions remain.\\n2. Run exactly this command once with `run_command`:\\n   `python skills/schema-portfolio/scripts/run_portfolio.py --output-dir schema_portfolio_outputs`\\n3. Continue only if the command succeeds and reports `status` equal to `PASS`.\\n4. Use `run_command` to print only the `candidates` array from `schema_portfolio_outputs/portfolio_manifest.json`.\\n5. In manifest order, call `submit_predictions` once for each listed CSV while submission budget remains. Record every successful submission ID and returned public score. Never submit any other file.\\n6. Sort successful submissions by returned public score, descending. Call `select_submission` exactly once with the IDs of the best two successful submissions (or the sole successful ID if only one succeeded). Do not guess IDs or scores.\\n7. Call `get_status` once more and report the selected IDs and their observed public scores.\\n\\n## Constraints\\n\\n- Do not write or edit modeling code.\\n- Do not inspect files other than the manifest after the portfolio command completes.\\n- Do not rerun training, tune parameters, or create extra candidates.\\n- Do not access the internet or install packages.\\n- Stop without submitting if the portfolio command fails or the manifest does not pass.\\n- Preserve enough of the limits below to complete selection:\\n  - submissions: {max_submissions}\\n  - selections: {max_selections}\\n  - tool calls: {max_tool_calls}\\n  - execution seconds per command: {max_exec_seconds}\\n  - total minutes: {max_time_minutes}\\n  - LLM calls: {max_llm_calls}\\n  - token budget: ${max_budget_usd}\\n\", \"skills/schema-portfolio/SKILL.md\": \"---\\nname: schema-portfolio\\ndescription: Runs a deterministic schema-aware binary tabular portfolio and emits validated prediction candidates with a JSON manifest.\\n---\\n\\n# Schema Portfolio\\n\\nUse this skill exactly once at the beginning of the evaluation.\\n\\n## Script\\n\\nRun:\\n\\n```bash\\npython skills/schema-portfolio/scripts/run_portfolio.py --output-dir schema_portfolio_outputs\\n```\\n\\nThe script reads only `train.csv`, `test.csv`, and `sample_submission.csv` from the working directory. It detects pandas string, object, categorical, and Boolean columns; tests an ordinal-aware representation; trains deterministic cross-validated CatBoost and scikit-learn models; and writes up to seven unique candidate CSVs.\\n\\nThe final stdout identifies `schema_portfolio_outputs/portfolio_manifest.json`. The manifest has `status: PASS`, model diagnostics, and a priority-ordered `candidates` list. Submit only the CSV paths in that list. Do not modify the script, generate additional modeling code, or invent filenames.\\n\", \"skills/schema-portfolio/scripts/run_portfolio.py\": \"\\\"\\\"\\\"Deterministic clean-room binary tabular portfolio.\\n\\nThe script is deliberately self-contained and only reads the three competition\\ninputs supplied in the working directory.  It writes validated prediction CSVs\\nplus a compact machine-readable manifest for the orchestration agent.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport re\\nimport time\\nfrom pathlib import Path\\nfrom typing import Callable\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostClassifier\\nfrom sklearn.compose import ColumnTransformer\\nfrom sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import roc_auc_score\\nfrom sklearn.model_selection import StratifiedKFold\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler\\n\\n\\nSEED = 20260729\\nORDINAL_PATTERN = re.compile(r\\\"^ord_(-?\\\\d+(?:\\\\.\\\\d+)?)$\\\")\\n\\n\\ndef utc_now() -> str:\\n    return pd.Timestamp.now(tz=\\\"UTC\\\").isoformat()\\n\\n\\ndef sha256_file(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\\"rb\\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\\"\\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef is_categorical(series: pd.Series) -> bool:\\n    dtype = series.dtype\\n    return bool(\\n        pd.api.types.is_object_dtype(dtype)\\n        or pd.api.types.is_string_dtype(dtype)\\n        or isinstance(dtype, pd.CategoricalDtype)\\n        or pd.api.types.is_bool_dtype(dtype)\\n    )\\n\\n\\ndef parse_binary_target(series: pd.Series) -> tuple[np.ndarray, list[str]]:\\n    if series.isna().any():\\n        raise ValueError(\\\"The training target contains missing values\\\")\\n    values = sorted(pd.unique(series), key=lambda value: str(value))\\n    if len(values) != 2:\\n        raise ValueError(f\\\"Expected a binary target, found {len(values)} classes\\\")\\n    encoded = (series == values[-1]).astype(np.int8).to_numpy()\\n    return encoded, [str(value) for value in values]\\n\\n\\ndef native_schema(\\n    train: pd.DataFrame, test: pd.DataFrame, features: list[str]\\n) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:\\n    x_train = train[features].copy()\\n    x_test = test[features].copy()\\n    categorical = [column for column in features if is_categorical(x_train[column])]\\n    numeric = [column for column in features if column not in categorical]\\n    for column in categorical:\\n        x_train[column] = x_train[column].fillna(\\\"__MISSING__\\\").astype(str)\\n        x_test[column] = x_test[column].fillna(\\\"__MISSING__\\\").astype(str)\\n    for column in numeric:\\n        x_train[column] = pd.to_numeric(x_train[column], errors=\\\"coerce\\\").replace(\\n            [np.inf, -np.inf], np.nan\\n        )\\n        x_test[column] = pd.to_numeric(x_test[column], errors=\\\"coerce\\\").replace(\\n            [np.inf, -np.inf], np.nan\\n        )\\n    return x_train, x_test, numeric, categorical\\n\\n\\ndef ordinal_schema(\\n    train: pd.DataFrame, test: pd.DataFrame, features: list[str]\\n) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], list[str]]:\\n    x_train, x_test, _, categorical = native_schema(train, test, features)\\n    parsed: list[str] = []\\n    for column in list(categorical):\\n        combined = pd.concat([x_train[column], x_test[column]], ignore_index=True)\\n        observed = combined[combined != \\\"__MISSING__\\\"]\\n        if observed.empty:\\n            continue\\n        extracted = observed.str.extract(ORDINAL_PATTERN, expand=False)\\n        if extracted.notna().all():\\n            x_train[column] = pd.to_numeric(\\n                x_train[column].str.extract(ORDINAL_PATTERN, expand=False), errors=\\\"coerce\\\"\\n            )\\n            x_test[column] = pd.to_numeric(\\n                x_test[column].str.extract(ORDINAL_PATTERN, expand=False), errors=\\\"coerce\\\"\\n            )\\n            parsed.append(column)\\n    categorical = [column for column in features if is_categorical(x_train[column])]\\n    numeric = [column for column in features if column not in categorical]\\n    return x_train, x_test, numeric, categorical, parsed\\n\\n\\ndef fold_count(y: np.ndarray) -> int:\\n    counts = np.bincount(y)\\n    minority = int(counts.min())\\n    if minority < 2:\\n        raise ValueError(\\\"Each target class needs at least two rows\\\")\\n    return min(5, minority)\\n\\n\\ndef catboost_family(\\n    x_train: pd.DataFrame,\\n    x_test: pd.DataFrame,\\n    y: np.ndarray,\\n    categorical: list[str],\\n    splits: list[tuple[np.ndarray, np.ndarray]],\\n) -> tuple[np.ndarray, np.ndarray, dict]:\\n    oof = np.zeros(len(x_train), dtype=float)\\n    test_predictions = np.zeros(len(x_test), dtype=float)\\n    best_iterations: list[int] = []\\n    for fold, (fit_index, valid_index) in enumerate(splits):\\n        model = CatBoostClassifier(\\n            iterations=800 if len(x_train) >= 2000 else 520,\\n            depth=7 if len(x_train) >= 2000 else 6,\\n            learning_rate=0.04,\\n            loss_function=\\\"Logloss\\\",\\n            eval_metric=\\\"AUC\\\",\\n            l2_leaf_reg=7.0,\\n            random_strength=0.30,\\n            random_seed=SEED + fold,\\n            verbose=False,\\n            allow_writing_files=False,\\n            thread_count=3,\\n        )\\n        model.fit(\\n            x_train.iloc[fit_index],\\n            y[fit_index],\\n            cat_features=categorical,\\n            eval_set=(x_train.iloc[valid_index], y[valid_index]),\\n            early_stopping_rounds=80,\\n            use_best_model=True,\\n            verbose=False,\\n        )\\n        oof[valid_index] = model.predict_proba(x_train.iloc[valid_index])[:, 1]\\n        test_predictions += model.predict_proba(x_test)[:, 1] / len(splits)\\n        best_iterations.append(int(model.get_best_iteration()))\\n    return oof, test_predictions, {\\\"best_iterations\\\": best_iterations}\\n\\n\\ndef logistic_pipeline(numeric: list[str], categorical: list[str]) -> Pipeline:\\n    transformer = ColumnTransformer(\\n        [\\n            (\\n                \\\"numeric\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"impute\\\", SimpleImputer(strategy=\\\"median\\\", add_indicator=True)),\\n                        (\\\"scale\\\", StandardScaler()),\\n                    ]\\n                ),\\n                numeric,\\n            ),\\n            (\\n                \\\"categorical\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"impute\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),\\n                        (\\n                            \\\"onehot\\\",\\n                            OneHotEncoder(handle_unknown=\\\"ignore\\\", min_frequency=2),\\n                        ),\\n                    ]\\n                ),\\n                categorical,\\n            ),\\n        ],\\n        sparse_threshold=0.3,\\n    )\\n    return Pipeline(\\n        [\\n            (\\\"features\\\", transformer),\\n            (\\n                \\\"model\\\",\\n                LogisticRegression(\\n                    C=0.3,\\n                    max_iter=2000,\\n                    solver=\\\"lbfgs\\\",\\n                    random_state=SEED,\\n                ),\\n            ),\\n        ]\\n    )\\n\\n\\ndef extra_trees_pipeline(\\n    numeric: list[str], categorical: list[str], seed: int\\n) -> Pipeline:\\n    transformer = ColumnTransformer(\\n        [\\n            (\\n                \\\"numeric\\\",\\n                SimpleImputer(strategy=\\\"median\\\", add_indicator=True),\\n                numeric,\\n            ),\\n            (\\n                \\\"categorical\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"impute\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),\\n                        (\\n                            \\\"ordinal\\\",\\n                            OrdinalEncoder(\\n                                handle_unknown=\\\"use_encoded_value\\\",\\n                                unknown_value=-1,\\n                                encoded_missing_value=-1,\\n                            ),\\n                        ),\\n                    ]\\n                ),\\n                categorical,\\n            ),\\n        ]\\n    )\\n    return Pipeline(\\n        [\\n            (\\\"features\\\", transformer),\\n            (\\n                \\\"model\\\",\\n                ExtraTreesClassifier(\\n                    n_estimators=500,\\n                    min_samples_leaf=2,\\n                    max_features=0.8,\\n                    class_weight=\\\"balanced\\\",\\n                    random_state=seed,\\n                    n_jobs=-1,\\n                ),\\n            ),\\n        ]\\n    )\\n\\n\\ndef hist_pipeline(numeric: list[str], categorical: list[str], seed: int) -> Pipeline:\\n    transformer = ColumnTransformer(\\n        [\\n            (\\\"numeric\\\", \\\"passthrough\\\", numeric),\\n            (\\n                \\\"categorical\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"impute\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),\\n                        (\\n                            \\\"ordinal\\\",\\n                            OrdinalEncoder(\\n                                handle_unknown=\\\"use_encoded_value\\\",\\n                                unknown_value=np.nan,\\n                                encoded_missing_value=np.nan,\\n                            ),\\n                        ),\\n                    ]\\n                ),\\n                categorical,\\n            ),\\n        ]\\n    )\\n    categorical_indices = list(range(len(numeric), len(numeric) + len(categorical)))\\n    return Pipeline(\\n        [\\n            (\\\"features\\\", transformer),\\n            (\\n                \\\"model\\\",\\n                HistGradientBoostingClassifier(\\n                    learning_rate=0.05,\\n                    max_iter=300,\\n                    max_leaf_nodes=15,\\n                    min_samples_leaf=20,\\n                    l2_regularization=2.0,\\n                    categorical_features=categorical_indices,\\n                    random_state=seed,\\n                    early_stopping=True,\\n                    validation_fraction=0.15,\\n                    n_iter_no_change=35,\\n                ),\\n            ),\\n        ]\\n    )\\n\\n\\ndef sklearn_family(\\n    x_train: pd.DataFrame,\\n    x_test: pd.DataFrame,\\n    y: np.ndarray,\\n    splits: list[tuple[np.ndarray, np.ndarray]],\\n    factory: Callable[[int], Pipeline],\\n) -> tuple[np.ndarray, np.ndarray, dict]:\\n    oof = np.zeros(len(x_train), dtype=float)\\n    test_predictions = np.zeros(len(x_test), dtype=float)\\n    for fold, (fit_index, valid_index) in enumerate(splits):\\n        model = factory(SEED + fold)\\n        model.fit(x_train.iloc[fit_index], y[fit_index])\\n        oof[valid_index] = model.predict_proba(x_train.iloc[valid_index])[:, 1]\\n        test_predictions += model.predict_proba(x_test)[:, 1] / len(splits)\\n    return oof, test_predictions, {}\\n\\n\\ndef rank01(values: np.ndarray) -> np.ndarray:\\n    return pd.Series(values).rank(method=\\\"average\\\", pct=True).to_numpy(dtype=float)\\n\\n\\ndef validate_output(\\n    output: pd.DataFrame,\\n    sample: pd.DataFrame,\\n    id_column: str,\\n    prediction_column: str,\\n) -> None:\\n    if list(output.columns) != list(sample.columns):\\n        raise ValueError(\\\"Output columns differ from the reference template\\\")\\n    if len(output) != len(sample):\\n        raise ValueError(\\\"Output row count differs from the reference template\\\")\\n    if output[id_column].duplicated().any():\\n        raise ValueError(\\\"Output identifiers are not unique\\\")\\n    if not output[id_column].equals(sample[id_column]):\\n        raise ValueError(\\\"Output identifier order differs from the reference template\\\")\\n    values = pd.to_numeric(output[prediction_column], errors=\\\"coerce\\\").to_numpy()\\n    if not np.isfinite(values).all():\\n        raise ValueError(\\\"Predictions contain non-finite values\\\")\\n    if ((values < 0.0) | (values > 1.0)).any():\\n        raise ValueError(\\\"Predictions fall outside [0, 1]\\\")\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\\"--train\\\", type=Path, default=Path(\\\"train.csv\\\"))\\n    parser.add_argument(\\\"--test\\\", type=Path, default=Path(\\\"test.csv\\\"))\\n    parser.add_argument(\\\"--sample\\\", type=Path, default=Path(\\\"sample_submission.csv\\\"))\\n    parser.add_argument(\\\"--output-dir\\\", type=Path, default=Path(\\\"schema_portfolio_outputs\\\"))\\n    args = parser.parse_args()\\n\\n    started = time.perf_counter()\\n    train = pd.read_csv(args.train)\\n    test = pd.read_csv(args.test)\\n    sample = pd.read_csv(args.sample)\\n    if len(sample.columns) != 2:\\n        raise ValueError(\\\"Expected a two-column reference template\\\")\\n    id_column, prediction_column = list(sample.columns)\\n    target_candidates = [column for column in train.columns if column not in test.columns]\\n    if len(target_candidates) != 1:\\n        raise ValueError(\\\"Could not identify exactly one train-only target column\\\")\\n    target_column = target_candidates[0]\\n    if target_column != prediction_column:\\n        raise ValueError(\\\"Training target and prediction column names differ\\\")\\n    if id_column not in test.columns:\\n        raise ValueError(\\\"Reference identifier is absent from test data\\\")\\n    if len(test) != len(sample):\\n        raise ValueError(\\\"Test and reference template row counts differ\\\")\\n    if not test[id_column].reset_index(drop=True).equals(sample[id_column].reset_index(drop=True)):\\n        raise ValueError(\\\"Test and reference identifier order differs\\\")\\n\\n    features = [column for column in test.columns if column != id_column]\\n    if not features:\\n        raise ValueError(\\\"No predictive features were found\\\")\\n    y, target_values = parse_binary_target(train[target_column])\\n    folds = fold_count(y)\\n    splitter = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)\\n    splits = list(splitter.split(train, y))\\n\\n    native_train, native_test, native_numeric, native_categorical = native_schema(\\n        train, test, features\\n    )\\n    ordinal_train, ordinal_test, ordinal_numeric, ordinal_categorical, parsed_ordinals = (\\n        ordinal_schema(train, test, features)\\n    )\\n\\n    predictions: dict[str, tuple[np.ndarray, np.ndarray]] = {}\\n    diagnostics: dict[str, dict] = {}\\n    failures: dict[str, str] = {}\\n\\n    def run_model(name: str, runner: Callable[[], tuple[np.ndarray, np.ndarray, dict]]) -> None:\\n        model_started = time.perf_counter()\\n        try:\\n            oof, test_prediction, extra = runner()\\n            if not np.isfinite(oof).all() or not np.isfinite(test_prediction).all():\\n                raise ValueError(\\\"Model returned non-finite predictions\\\")\\n            predictions[name] = (oof, np.clip(test_prediction, 1e-7, 1.0 - 1e-7))\\n            diagnostics[name] = {\\n                \\\"oof_auc\\\": float(roc_auc_score(y, oof)),\\n                \\\"runtime_seconds\\\": time.perf_counter() - model_started,\\n                **extra,\\n            }\\n        except Exception as exc:\\n            failures[name] = f\\\"{type(exc).__name__}: {exc}\\\"\\n\\n    run_model(\\n        \\\"catboost_native\\\",\\n        lambda: catboost_family(\\n            native_train, native_test, y, native_categorical, splits\\n        ),\\n    )\\n    if parsed_ordinals:\\n        run_model(\\n            \\\"catboost_ordinal\\\",\\n            lambda: catboost_family(\\n                ordinal_train, ordinal_test, y, ordinal_categorical, splits\\n            ),\\n        )\\n\\n    run_model(\\n        \\\"logistic\\\",\\n        lambda: sklearn_family(\\n            ordinal_train,\\n            ordinal_test,\\n            y,\\n            splits,\\n            lambda _seed: logistic_pipeline(ordinal_numeric, ordinal_categorical),\\n        ),\\n    )\\n    run_model(\\n        \\\"extra_trees\\\",\\n        lambda: sklearn_family(\\n            ordinal_train,\\n            ordinal_test,\\n            y,\\n            splits,\\n            lambda seed: extra_trees_pipeline(ordinal_numeric, ordinal_categorical, seed),\\n        ),\\n    )\\n    run_model(\\n        \\\"hist_gradient_boosting\\\",\\n        lambda: sklearn_family(\\n            ordinal_train,\\n            ordinal_test,\\n            y,\\n            splits,\\n            lambda seed: hist_pipeline(ordinal_numeric, ordinal_categorical, seed),\\n        ),\\n    )\\n\\n    if len(predictions) < 2:\\n        raise RuntimeError(f\\\"Fewer than two model families succeeded: {failures}\\\")\\n\\n    ranking = sorted(\\n        predictions,\\n        key=lambda name: diagnostics[name][\\\"oof_auc\\\"],\\n        reverse=True,\\n    )\\n    for count in (2, 3):\\n        if len(ranking) < count:\\n            continue\\n        members = ranking[:count]\\n        oof_blend = np.mean([rank01(predictions[name][0]) for name in members], axis=0)\\n        test_blend = np.mean([rank01(predictions[name][1]) for name in members], axis=0)\\n        name = f\\\"rank_top{count}\\\"\\n        predictions[name] = (oof_blend, test_blend)\\n        diagnostics[name] = {\\n            \\\"oof_auc\\\": float(roc_auc_score(y, oof_blend)),\\n            \\\"runtime_seconds\\\": 0.0,\\n            \\\"members\\\": members,\\n        }\\n\\n    submission_priority = [\\n        \\\"rank_top2\\\",\\n        \\\"catboost_native\\\",\\n        \\\"catboost_ordinal\\\",\\n        \\\"rank_top3\\\",\\n        \\\"logistic\\\",\\n        \\\"hist_gradient_boosting\\\",\\n        \\\"extra_trees\\\",\\n    ]\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    candidates: list[dict] = []\\n    accepted_vectors: list[np.ndarray] = []\\n    for name in submission_priority:\\n        if name not in predictions:\\n            continue\\n        vector = predictions[name][1]\\n        if any(np.allclose(vector, prior, rtol=0.0, atol=1e-12) for prior in accepted_vectors):\\n            diagnostics[name][\\\"omitted_as_duplicate\\\"] = True\\n            continue\\n        accepted_vectors.append(vector.copy())\\n        output = sample.copy()\\n        output[prediction_column] = vector\\n        validate_output(output, sample, id_column, prediction_column)\\n        path = args.output_dir / f\\\"{name}.csv\\\"\\n        output.to_csv(path, index=False)\\n        candidates.append(\\n            {\\n                \\\"name\\\": name,\\n                \\\"path\\\": str(path),\\n                \\\"sha256\\\": sha256_file(path),\\n                \\\"oof_auc\\\": diagnostics[name][\\\"oof_auc\\\"],\\n            }\\n        )\\n\\n    manifest = {\\n        \\\"status\\\": \\\"PASS\\\",\\n        \\\"completed_at_utc\\\": utc_now(),\\n        \\\"script_sha256\\\": sha256_file(Path(__file__)),\\n        \\\"input_sha256\\\": {\\n            \\\"train\\\": sha256_file(args.train),\\n            \\\"test\\\": sha256_file(args.test),\\n            \\\"sample\\\": sha256_file(args.sample),\\n        },\\n        \\\"train_rows\\\": len(train),\\n        \\\"test_rows\\\": len(test),\\n        \\\"feature_count\\\": len(features),\\n        \\\"numeric_count_native\\\": len(native_numeric),\\n        \\\"categorical_count_native\\\": len(native_categorical),\\n        \\\"categorical_columns_native\\\": native_categorical,\\n        \\\"parsed_ordinal_columns\\\": parsed_ordinals,\\n        \\\"target_values\\\": target_values,\\n        \\\"folds\\\": folds,\\n        \\\"ranking\\\": ranking,\\n        \\\"diagnostics\\\": diagnostics,\\n        \\\"failures\\\": failures,\\n        \\\"candidates\\\": candidates,\\n        \\\"runtime_seconds\\\": time.perf_counter() - started,\\n    }\\n    manifest_path = args.output_dir / \\\"portfolio_manifest.json\\\"\\n    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n    print(\\n        json.dumps(\\n            {\\n                \\\"status\\\": manifest[\\\"status\\\"],\\n                \\\"manifest\\\": str(manifest_path),\\n                \\\"candidate_count\\\": len(candidates),\\n                \\\"candidate_paths\\\": [item[\\\"path\\\"] for item in candidates],\\n                \\\"failures\\\": failures,\\n                \\\"runtime_seconds\\\": manifest[\\\"runtime_seconds\\\"],\\n            },\\n            sort_keys=True,\\n        )\\n    )\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\"}")
SOURCE_HASHES = json.loads("{\"agent.yaml\": \"b241986ffc52ff5e5fbbcacba72e5bfbbe48ab63dedb7db383fd3b99add9eec5\", \"configs/sampling.yaml\": \"5e7668cd9877a77eeb8a84468e92c582523eb168ad165a7f0a44008200e0b6e8\", \"prompts/system.md\": \"83300abf7da8d5a4629036c91b345c68387debaef9e5b33eec77a7c96d6456eb\", \"skills/schema-portfolio/SKILL.md\": \"3c57c823eaf900ae29a1b92f17617dfc672e379ac21a21ad38a2b93f5833a316\", \"skills/schema-portfolio/scripts/run_portfolio.py\": \"8ac114fccb4ad44ebda72dfa98d73d637c493bb917dc48279163f75385a28f0c\"}")
EVIDENCE_HASHES = json.loads("{\"cleanroom_candidates\": \"88fa1b44574d6c005db45778c2a7566b79fd81d9a9deb77717e46b7296edebfa\", \"cleanroom_receipt\": \"749d8530cd24125f89c50f076bbba69f6e2f418de792b15aee105e8ad7ad165f\", \"cleanroom_selection\": \"fc63b6aa233fd8341f09f9196da5eef176cbaa82f517aed99ad4ee4136012df6\", \"dtype_repaired_baseline\": \"4a30f32ac0876180d29f07e510c256e5163c39a7c39997dcdb90618ee6b2dc36\", \"raw_reference_replay\": \"e51a2a86f7ef9ca901c9ed242f005543c93979f08d551e320dbd2bc314fb8407\", \"schema_probe\": \"83559c56a53445a9eb00e892b108e9469b3f19318e8961c1f8178e2d3bdd9150\"}")

In [ ]:
base = pd.DataFrame(BASELINE)
raw = pd.DataFrame(RAW_BASELINE)
clean = pd.DataFrame(CLEANROOM)
candidates = pd.DataFrame(CANDIDATES)
schemas = pd.DataFrame(SCHEMAS)

raw_mean = float(raw.final_private_auc.mean())
repaired_mean = float(base.final_private_auc.mean())
clean_mean = float(clean.final_private_auc.mean())
delta = clean_mean - repaired_mean
cards = [
    ("16", "official local tasks"),
    (f"{raw_mean:.6f}", "unrepaired replay"),
    (f"{repaired_mean:.6f}", "dtype-repaired reference"),
    (f"{clean_mean:.6f}", "clean-room portfolio"),
    (f"{delta:+.6f}", "matched local delta"),
]
html = '<div style="display:flex;gap:12px;flex-wrap:wrap;margin:12px 0 20px">'
for value, label in cards:
    html += f'<div style="min-width:155px;padding:16px 18px;border-radius:14px;background:#F5F8FF;border:1px solid #D8E3F5"><div style="font-size:25px;font-weight:800;color:#176BFF">{value}</div><div style="font-size:12px;color:#52627A">{label}</div></div>'
html += '</div>'
display(HTML(html))

## 1. The failure was schema inference, not model capacity

Pandas 3 may infer textual CSV columns as `str`, while older checks often looked only for `object`, `category`, or `bool`. A text column that escapes categorical detection is then pushed into a numeric path, coerced to missing values, and can collapse a whole model family. The repair is small but consequential:

```python
is_categorical = (
    pd.api.types.is_object_dtype(dtype)
    or pd.api.types.is_string_dtype(dtype)   # the critical guard
    or isinstance(dtype, pd.CategoricalDtype)
    or pd.api.types.is_bool_dtype(dtype)
)
```

The popular reference is used only as an external, attributed replay target. Its source is not included in this notebook or archive.

In [ ]:
labels = ["Unrepaired\nreference", "Dtype-repaired\nreference", "Clean-room\nportfolio"]
values = [raw_mean, repaired_mean, clean_mean]
fig, ax = plt.subplots(figsize=(9.2, 4.8))
bars = ax.bar(labels, values, color=[COLORS["red"], COLORS["gold"], COLORS["blue"]], width=.62)
ax.set_ylim(max(0, min(values)-.025), max(values)+.012)
ax.set_ylabel("Mean selected-final Private AUC\n(16 official local tasks)")
ax.set_title("One dtype guard recovers the collapsed categorical tasks")
for bar, value in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, value+.0012, f"{value:.6f}", ha="center", fontweight="bold")
ax.text(1, repaired_mean-.012, f"+{repaired_mean-raw_mean:.4f}", ha="center", color=COLORS["ink"], fontweight="bold")
sns.despine(ax=ax)
plt.show()

In [ ]:
merged = base[["dataset","final_private_auc"]].rename(columns={"final_private_auc":"dtype_repaired"}).merge(
    clean[["dataset","final_private_auc"]].rename(columns={"final_private_auc":"clean_room"}), on="dataset"
)
merged["delta"] = merged.clean_room - merged.dtype_repaired
merged = merged.sort_values("delta")
y = np.arange(len(merged))
fig, ax = plt.subplots(figsize=(10, 7.2))
ax.hlines(y, merged.dtype_repaired, merged.clean_room, color="#CAD5E5", lw=3)
ax.scatter(merged.dtype_repaired, y, s=58, color=COLORS["gold"], label="dtype-repaired reference", zorder=3)
ax.scatter(merged.clean_room, y, s=62, color=COLORS["blue"], label="clean-room portfolio", zorder=4)
ax.set_yticks(y, merged.dataset.str.replace("train_", "Task "))
ax.axvline(repaired_mean, color=COLORS["gold"], ls="--", alpha=.45)
ax.axvline(clean_mean, color=COLORS["blue"], ls="--", alpha=.45)
ax.set_xlabel("Selected-final Private AUC (offline)")
ax.set_title("Matched per-task audit: improvements must survive all 16 datasets")
ax.legend(frameon=False, loc="lower right")
sns.despine(ax=ax)
plt.show()
display(merged.style.format({"dtype_repaired":"{:.6f}","clean_room":"{:.6f}","delta":"{:+.6f}"}).background_gradient(subset=["delta"], cmap="RdYlGn"))

## 2. A portfolio that respects schema and selection risk

The archive does not ask an LLM to improvise modeling code. It runs one frozen, deterministic script, validates every candidate CSV, submits the small portfolio, and selects the two observed public leaders. The script combines native categorical boosting, ordinal-aware preprocessing, linear and tree diversity, and rank ensembles. Model failures are isolated and recorded instead of silently poisoning the run.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.axis("off")
nodes = [(0.03,"Schema\naudit"),(0.23,"5-fold OOF\nportfolio"),(0.45,"Rank\nensembles"),(0.66,"CSV + hash\nvalidation"),(0.86,"Public top-two\nselection")]
for x, label in nodes:
    ax.add_patch(plt.Rectangle((x,.32),.13,.36,transform=ax.transAxes,facecolor=COLORS["mist"],edgecolor=COLORS["blue"],lw=1.8))
    ax.text(x+.065,.50,label,transform=ax.transAxes,ha="center",va="center",fontweight="bold",color=COLORS["ink"])
for (x,_),(nx,_) in zip(nodes,nodes[1:]):
    ax.annotate("",xy=(nx,.50),xytext=(x+.13,.50),xycoords=ax.transAxes,textcoords=ax.transAxes,arrowprops=dict(arrowstyle="->",lw=2,color=COLORS["cyan"]))
ax.set_title("Frozen execution path inside the submitted agent", pad=10)
plt.show()

In [ ]:
schema_plot = schemas.set_index("dataset")[["numeric","categorical"]]
ax = schema_plot.plot(kind="bar", stacked=True, figsize=(11,4.8), color=[COLORS["blue"],COLORS["gold"]], width=.78)
ax.set_title("The 16 tasks span pure numeric, mixed, and pure categorical schemas")
ax.set_ylabel("Feature count")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False, ncol=2)
sns.despine(ax=ax)
plt.show()

In [ ]:
heat = candidates.pivot(index="dataset", columns="candidate", values="private_auc")
fig, ax = plt.subplots(figsize=(12,6.8))
sns.heatmap(heat, cmap="viridis", vmin=max(.5,float(np.nanmin(heat.values))), vmax=float(np.nanmax(heat.values)), linewidths=.4, linecolor="white", ax=ax, cbar_kws={"label":"Private AUC (offline)"})
ax.set_title("Candidate diversity: no single family wins every task")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()

In [ ]:
clean["selection_headroom"] = clean.oracle_best_private_auc - clean.final_private_auc
ordered = clean.sort_values("selection_headroom", ascending=False)
fig, ax = plt.subplots(figsize=(10,4.8))
colors = [COLORS["red"] if value > .002 else COLORS["cyan"] for value in ordered.selection_headroom]
ax.bar(ordered.dataset, ordered.selection_headroom, color=colors)
ax.axhline(0, color=COLORS["ink"], lw=1)
ax.set_title("Selection risk: oracle Private minus selected-final Private")
ax.set_ylabel("AUC headroom (offline)")
ax.tick_params(axis="x", rotation=45)
sns.despine(ax=ax)
plt.show()

## 3. Public ecology, provenance, and what is *not* claimed

Votes and score-sort position measure different things. This small snapshot is included to motivate a notebook that is both reproducible and useful—not to infer exact leaderboard scores. The score axis below is **rank only** because Kaggle's listing did not expose exact official scores.

In [ ]:
eco = pd.DataFrame(ECOSYSTEM)
fig, ax = plt.subplots(figsize=(8.8,5.5))
sizes = 60 + eco.votes*4
ax.scatter(eco.vote_rank, eco.score_rank, s=sizes, c=eco.votes, cmap="Blues", edgecolor="white", linewidth=1.2)
for row in eco.itertuples():
    ax.annotate(row.notebook, (row.vote_rank,row.score_rank), xytext=(5,4), textcoords="offset points", fontsize=8)
ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel("Vote rank (1 is highest)")
ax.set_ylabel("Kaggle score-sort rank (1 is first; exact score unavailable)")
ax.set_title("Popularity and score-sort order are not interchangeable")
sns.despine(ax=ax)
plt.show()

In [ ]:
prov = pd.DataFrame(PROVENANCE)
prov["source"] = [f'<a href="{url}" target="_blank">{name}</a>' for name,url in zip(prov.source,prov.url)]
display(HTML(prov.drop(columns="url").to_html(index=False, escape=False)))
display(Markdown("**License discipline:** a null Kaggle metadata license is treated as unknown permission. No public-notebook source code is present in this archive."))

## 4. Build the exact submission archive

The next cell writes only the five audited clean-room files embedded in this notebook. It refuses path traversal, symlinks, missing root config, forbidden evaluation tokens, invalid Python, and unexpected ZIP members. ZIP timestamps are fixed, making the archive byte-reproducible.

In [ ]:
def sha256_path(path):
    digest = hashlib.sha256()
    with open(path,"rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
if (out_root / "submission.zip").exists():
    raise RuntimeError("Refusing to overwrite an existing submission.zip")
build_root = Path(tempfile.mkdtemp(prefix="schema_agent_build_", dir=out_root))
agent_root = build_root / "agent"
agent_root.mkdir()

for relative, content in SOURCES.items():
    relative_path = Path(relative)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError(f"Unsafe source path: {relative}")
    destination = agent_root / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")

assert (agent_root / "agent.yaml").is_file()
assert not any(path.is_symlink() for path in agent_root.rglob("*"))
for path in agent_root.rglob("*.py"):
    py_compile.compile(str(path), doraise=True)
for path in agent_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".yaml",".yml",".md",".py"}:
        lower = path.read_text(encoding="utf-8").lower()
        assert "solution.csv" not in lower
        assert "private_auc" not in lower
        assert "public_auc" not in lower
        assert "../" not in lower

archive_path = out_root / "submission.zip"
members = sorted(path for path in agent_root.rglob("*") if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc")
with zipfile.ZipFile(archive_path,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for path in members:
        relative = path.relative_to(agent_root).as_posix()
        info = zipfile.ZipInfo(relative, date_time=(2026,1,1,0,0,0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, path.read_bytes())

expected = sorted(path.relative_to(agent_root).as_posix() for path in members)
with zipfile.ZipFile(archive_path) as archive:
    actual = sorted(archive.namelist())
    assert actual == expected
    assert actual[0] == "agent.yaml"
    assert all(not name.startswith("/") and ".." not in Path(name).parts for name in actual)
    assert all(not (info.external_attr >> 16) & 0o170000 == 0o120000 for info in archive.infolist())

archive_sha = sha256_path(archive_path)
build_receipt = {
    "status":"PASS",
    "scope":"clean-room archive build; offline evidence is not an official Kaggle score",
    "archive":str(archive_path),
    "archive_sha256":archive_sha,
    "members":actual,
    "member_sha256":{name:sha256_path(agent_root/name) for name in actual},
    "source_hashes":SOURCE_HASHES,
    "evidence_hashes":EVIDENCE_HASHES,
    "no_symlinks":True,
    "no_path_traversal":True,
    "no_evaluation_tokens":True,
    "python_compile":"PASS",
}
receipt_out = out_root / "PUBLIC_NOTEBOOK_AUDIT_RECEIPT.json"
receipt_out.write_text(json.dumps(build_receipt,indent=2,sort_keys=True)+"\n",encoding="utf-8")
display(Markdown(f"### ✅ Archive PASS — `{archive_path.name}`\n\nSHA256: `{archive_sha}`"))
display(pd.DataFrame({"archive member":actual,"sha256":[build_receipt["member_sha256"][name] for name in actual]}))

In [ ]:
gates = pd.DataFrame([
    ("Root agent.yaml", True),
    ("No symlinks", build_receipt["no_symlinks"]),
    ("No path traversal", build_receipt["no_path_traversal"]),
    ("No solution/public/private token", build_receipt["no_evaluation_tokens"]),
    ("All Python compiles", build_receipt["python_compile"] == "PASS"),
    ("16-task matched local gate", RECEIPT["decision"] == "PROMOTE"),
    ("Positive vs repaired reference", RECEIPT["delta_vs_dtype_repaired_public_baseline"] > 0),
], columns=["gate","pass"])
display(gates.style.map(lambda value: "background:#DFF7EA;color:#086C3C;font-weight:bold" if value is True else "", subset=["pass"]))

## Reproducibility receipt

- Modeling source is embedded above and hashed file-by-file.
- The agent reads only `train.csv`, `test.csv`, and `sample_submission.csv`.
- Evaluation solutions were used only by an external scorer after predictions were frozen.
- Public notebooks are credited for research orientation; no unlicensed source was copied.
- `submission.zip` and `PUBLIC_NOTEBOOK_AUDIT_RECEIPT.json` are downloadable notebook outputs.

**Next evidence boundary:** only a successful Kaggle notebook run, exact downloaded-output validation, and a COMPLETE official row with a nonempty score could support an official score claim. This notebook intentionally makes no such claim.